In [1]:
import pandas as pd
import numpy as np
import os

file_path = "datasets/madalena_processed_30min"

def feature_engineer(file_path):
    df = pd.read_csv(file_path)
    df = df.loc[:, ~df.columns.str.startswith('Unnamed:')]    
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)
    df = df.sort_index()
    df['day_of_year'] = df.index.dayofyear
    df['season'] = df.index.month % 12 // 3
    df['month'] = df.index.month
    df['day_of_week'] = df.index.weekday
    df['hour'] = df.index.hour
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 48)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 48)

    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['is_weekend'] = (df.index.weekday >= 5).astype(int)
    df['energy_lag_48'] = df['total_energy'].shift(48).fillna(method='bfill')
    df['energy_lag_96'] = df['total_energy'].shift(96).fillna(method='bfill')
    df['energy_lag_1'] = df['total_energy'].shift(1).fillna(method='bfill')
    df['energy_diff'] = df['total_energy'].diff().fillna(method='bfill')
    df['max_energy'] = df['total_energy'].rolling(window=48).max().fillna(method='bfill')
    df['min_energy'] = df['total_energy'].rolling(window=48).min().fillna(method='bfill')
    df['energy_diff_daily'] = df['max_energy'] - df['min_energy'] 
    df['energy_rolling'] = df['total_energy'].rolling(window=1460).mean().fillna(method='bfill')
    df['Tout_diff'] = df['Tout'].diff().fillna(method="bfill")
    df['Tout_lag_1'] = df['Tout'].shift(1).fillna(method='bfill')
    df['Tin_lag_1'] = df['Tin'].shift(1).fillna(method='bfill')
    df['is_heating_season'] = df['month'].isin([11, 12, 1, 2, 3]).astype(int)

    df['Tout_lag_2'] = df['Tout'].shift(2).fillna(method='bfill')
    df['energy_std_48'] = df['total_energy'].rolling(window=48).std().fillna(method='bfill')
    df['energy_spike'] = (df['total_energy'] > df['total_energy'].rolling(window=48).mean().fillna(method='bfill') + 3*df['total_energy'].rolling(window=48).std().fillna(method='bfill')).astype(float)
    df['rolling_max_3h'] = df['total_energy'].rolling(window=6).max().fillna(method='bfill')
    df['recent_spike'] = df['total_energy'].diff().abs().rolling(2).max() > 200

    return df

In [2]:
import torch
from torch.utils.data import TensorDataset, DataLoader


def create_mimo_sequences(X_data, y_data, input_len=96, output_len=48):
    X, y = [], []
    for i in range(len(X_data) - input_len):
        X.append(X_data[i:i+input_len])
        y.append(y_data[i+input_len]) 
        # spike_targets.append(spike_data[i+input_len:i+input_len+output_len])
    return np.array(X), np.array(y)

In [3]:
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import seaborn as sns
import calendar
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def denormalize(norm_vals, original_min, original_max):
    return norm_vals * (original_max - original_min) + original_min

for file in os.listdir(file_path):
    full_path = os.path.join(file_path, file)
    if file == 'E_152.csv':
        continue
    df = feature_engineer(full_path)
    # energy_min = df['total_energy'].min()
    # energy_max = df['total_energy'].max()
    # temp_min = df['Tout'].min()
    # temp_max = df['Tout'].max()
    # for col in ['total_energy', 'Tout']:
    #     min_val = df[col].min()
    #     max_val = df[col].max()
    #     df[col] = (df[col] - min_val) / (max_val - min_val)
    # monthly_groups = df['total_energy'].groupby(df.index.month)

    # min_len = min(len(group) for _, group in monthly_groups)

    # monthly_profiles = np.vstack([
    #     group.values[:min_len] for _, group in monthly_groups
    # ])

    # k = 4
    # kmeans = KMeans(n_clusters=k, random_state=42)
    # clusters = kmeans.fit_predict(monthly_profiles)
    # available_months = sorted(df.index.month.unique())
    # month_cluster_map = dict(zip(available_months, clusters))
    # df['cluster'] = df.index.month.map(month_cluster_map)
    # for cluster_id in np.unique(clusters):
    #     cluster_months = [m for m, c in month_cluster_map.items() if c == cluster_id]
    #     plt.figure(figsize=(12, 4))
        
    #     for m in cluster_months:
    #         energy = df[df.index.month == m]['total_energy'].values[:min_len]  
    #         plt.plot(energy, label=f'Month {m}')
        
    #     plt.title(f'Energy Profiles - Cluster {cluster_id}')
    #     plt.xlabel('Time Index')
    #     plt.ylabel('Normalized Energy')
    #     plt.legend()
    #     plt.grid(True)
    #     plt.show()

    features = ['total_energy', 'Tout', 'hour_sin', 'hour_cos', 'is_weekend', 'day_of_week', 'Tin', 'Tin_lag_1', 'RH', 'energy_rolling']
    df['rolling_mean_6'] = df['total_energy'].rolling(8).mean().fillna(method='bfill')
    df['rolling_std_6'] = df['total_energy'].rolling(8).std().fillna(method='bfill')
    df['energy_lag_336'] = df['total_energy'].shift(336).fillna(method='bfill')
    df['Tout_mean'] = df['Tout'].rolling(6).mean().fillna(method='bfill')
    # df['energy_lag_336'] = (df['energy_lag_336'] - energy_min) / (energy_max - energy_min)
    # df['energy_lag_48'] = (df['energy_lag_48'] - energy_min) / (energy_max - energy_min)
    # df['energy_diff'] = (df['energy_diff'] - energy_min) / (energy_max - energy_min)
    # df['max_energy'] = (df['max_energy'] - energy_min) / (energy_max - energy_min)
    # df['Tout_diff'] = (df['Tout_diff'] - temp_min) / (temp_max - temp_min)
    # df['Tout_mean'] = (df['Tout_mean'] - temp_min) / (temp_max - temp_min)
    # df['rolling_std_6'] = (df['rolling_std_6'] - energy_min) / (energy_max - energy_min)
    # df['rolling_mean_6'] = (df['rolling_mean_6'] - energy_min) / (energy_max - energy_min)
    continuous_features = ['total_energy', 'Tout', 'Tin', 'RH', 'Tin_lag_1']

    scaler_cont = StandardScaler()
    df[continuous_features] = scaler_cont.fit_transform(df[continuous_features])
    data = df[features].values.astype(np.float32)
    target = df['heat_pump_active'].values.reshape(-1, 1).astype(np.float32)

    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    # data_scaled = scaler_x.fit_transform(data)
    # target = data[:, 0].reshape(-1, 1)  
    # target_scaled = scaler_y.fit_transform(target)
    X, y = create_mimo_sequences(data, target, input_len=96)
    X = X.astype(np.float32)
    y = y.astype(np.float32)
    
    split_idx = int(0.9 * len(X))
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, shuffle=False, random_state=42)

    # Convert to tensors
    # X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    # y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    # X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    # y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

    # train_ds = TensorDataset(X_train_tensor, y_train_tensor)
    # test_ds = TensorDataset(X_test_tensor, y_test_tensor)
    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
    class KCNNLSTM(nn.Module):
        def __init__(self, input_features, conv_out=64, lstm_hidden=512, output_len=48):
            super(KCNNLSTM, self).__init__()
            # self.conv = nn.Conv1d(input_features, conv_out, kernel_size=2, padding='same')
            # self.relu = nn.ReLU()
            self.lstm = nn.LSTM(input_features, lstm_hidden, batch_first=True)
            # self.dropout = nn.Dropout(0.2)
            self.fc = nn.Linear(lstm_hidden, 1)
            nn.init.xavier_uniform_(self.fc.weight)
            nn.init.zeros_(self.fc.bias)
            # self.fc1 = nn.Linear(lstm_hidden, 64)
            # self.fc2 = nn.Linear(64, output_len)

        def forward(self, x):
            # x = x.permute(0, 2, 1)  
            # x = self.conv(x)
            # x = self.relu(x)
            # x = x.permute(0, 2, 1)  
            lstm_out, _ = self.lstm(x)
            x = lstm_out[:, -1, :]
            # x = self.dropout(x)
            return self.fc(x)
            # x = self.fc1(x)
            # x = self.fc2(x)
            # return x
        
    model = KCNNLSTM(input_features=X.shape[2]).to(device)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)
    n_epochs = 50
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        batch_count = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            preds = model(xb)
            loss = loss_fn(preds, yb)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            batch_count += 1

        avg_loss = total_loss / batch_count
        print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {avg_loss:.6f}")

        # # Validation
        # model.eval()
        # val_losses = []
        # with torch.no_grad():
        #     for xb, yb in test_loader:
        #         xb, yb = xb.to(device), yb.to(device).unsqueeze(1)
        #         preds = model(xb)
        #         loss = loss_fn(preds, yb)
        #         val_losses.append(loss.item())

        # avg_train_loss = np.mean(train_losses)
        # avg_val_loss = np.mean(val_losses)
        # print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")
    model.eval()
    all_preds = []
    all_true = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            preds = model(xb)
            all_preds.append(preds.cpu())
            all_true.append(yb)

    # Concatenate all batches
    y_pred = torch.cat(all_preds, dim=0)
    y_true = torch.cat(all_true, dim=0)

    # Apply sigmoid to logits
    y_pred_probs = torch.sigmoid(y_pred).numpy()
    y_true = y_true.numpy()

    # Threshold at 0.5
    y_pred_classes = (y_pred_probs >= 0.5).astype(int)

    # Metrics
    acc = accuracy_score(y_true, y_pred_classes)
    prec = precision_score(y_true, y_pred_classes)
    rec = recall_score(y_true, y_pred_classes)
    print(f"\nEvaluation:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    plt.figure(figsize=(12, 5))
    plt.plot(y_true[:20], label="True (Heat Pump Active)", marker='o')
    plt.plot(y_pred_classes[:20], label="Predicted", marker='x')
    plt.legend()
    plt.grid()
    plt.title("First 100 test predictions")
    plt.show()

C:\Users\Kelsier\AppData\Local\Temp\ipykernel_13280\2284308995.py:24: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['energy_lag_48'] = df['total_energy'].shift(48).fillna(method='bfill')
C:\Users\Kelsier\AppData\Local\Temp\ipykernel_13280\2284308995.py:25: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['energy_lag_96'] = df['total_energy'].shift(96).fillna(method='bfill')
C:\Users\Kelsier\AppData\Local\Temp\ipykernel_13280\2284308995.py:26: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['energy_lag_1'] = df['total_energy'].shift(1).fillna(method='bfill')
C:\Users\Kelsier\AppData\Local\Temp\ipykernel_13280\2284308995.py:27: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Us

KeyboardInterrupt: 